# 5.7. Predicting House Prices on Kaggle

Let's put our deep learning skills to the test by training a machine learning model on the [Ames Housing Dataset](https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset).

Our data pre-processing pipeline is described below.

For the features:

1. Remove the redundant ID column
1. Compute the mean for each column of numerical data, skipping NA values from the mean computation
1. Replace NA values in numerical data with the mean value from their corresponding column
1. Apply StandardScaler to numerical columns to have zero mean and unit variance
1. Apply one-hot encoding to categorical \(non-numerical\) columns with the following rules:
    1. Treat NA values as its own distinct category
    1. Remove the 1st category from the generated one-hot encoding to avoid collinearity

For the labels:

1. Apply `log1p` transformation to the sale price
1. Apply StandardScaler to \(1\) so the transformed labels have zero mean and unit variance

The dataset we expect to get after pre-processing as below.

1. 1460 training samples
1. Approximately 331 input features - generated from the original 80 input features after applying one-hot encoding
1. 1 output label corresponding to the house price

Since we have limited, high-dimensional data, we'll use $k$-fold cross validation with $k = 10$.

We'll train our neural network with the architecture below over 100 epochs with a batch size of $2 ^ 6 = 64$.

1. 1st hidden layer: 331 input channels, 128 output channels, ReLU activation
1. Dropout layer with `p=0.2`
1. 2nd hidden layer: 128 input channels, 64 output channels, ReLU activation
1. Final fully connected layer with 64 input channels and 1 output channel. The output channel is the \(transformed\) predicted sale price

With the outputs from our model, we plan to apply the following inverse transformations to obtain meaningful predicted housing sale prices in USD.

1. Apply the inverse of StandardScaler to recover the mean and variance of the `log1p` transformed housing prices based on the training data
1. Apply `expm1` to the results in \(1\) to undo the `log1p` transformation and obtain the actual predicted housing price in USD

Our choice of loss function, optimizer and hyperparameters below.

1. Loss function: MSE loss on the \(transformed\) predictions
1. Optimizer: minibatch SGD with batches of size 64
1. Learning rate: 0.01
1. Weight decay: `1e-4`
1. Momentum: `0.9`

In [1]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 5.7.1. Downloading Data

We'll download a copy of the Ames housing dataset from [D2L](https://d2l.ai/)-owned S3 bucket\(s\).

In [2]:
import os

dataset_dir = 'data/ames/'
os.makedirs(dataset_dir, exist_ok=True)

In [3]:
import urllib.request

train_ds_link = 'http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_train.csv'
test_ds_link = 'http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_test.csv'
train_ds_path = os.path.join(dataset_dir, 'kaggle_house_pred_train.csv')
test_ds_path = os.path.join(dataset_dir, 'kaggle_house_pred_test.csv')

with urllib.request.urlopen(train_ds_link) as response:
    with open(train_ds_path, 'wb') as file:
        file.write(response.read())

with urllib.request.urlopen(test_ds_link) as response:
    with open(test_ds_path, 'wb') as file:
        file.write(response.read())

## 5.7.2. Kaggle

We'll submit our results to this Kaggle competition: [House Prices: Advanced Regression Techniques | Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)

## 5.7.3. Accessing and Reading the Dataset

Let's load our data from the downloaded CSV datasets using [Pandas](https://pandas.pydata.org/) and inspect their shapes. First, we'll need to install Pandas 3.0.2 with `pip`.

In [4]:
%pip install pandas==3.0.2


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Now load the data with [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html#pandas.read_csv) and inspect their shapes.

In [5]:
import pandas as pd

# "df" stands for Pandas DataFrame
train_df = pd.read_csv(train_ds_path)
test_df = pd.read_csv(test_ds_path)
train_df.shape, test_df.shape

((1460, 81), (1459, 80))

The training set has 1460 samples while the test set has 1459 samples. Both sets have 80 input features. Additionally, the training set has 1 output label which is the house price in USD.

Next, let's inspect our data in detail and pre-process it in a form suitable for feeding our deep network for training. As we'll see shortly, not all features are relevant and some data is missing - we'll see how to deal with these in a moment to obtain clean training and test data.

## 5.7.4. Data Preprocessing

Let's take a look at the first 4 and final 2 input features as well as the output label from the training set. We'll take just the first 4 samples.

In [6]:
print(train_df.iloc[:4, [0, 1, 2, 3, -3, -2, -1]])

   Id  MSSubClass MSZoning  LotFrontage SaleType SaleCondition  SalePrice
0   1          60       RL         65.0       WD        Normal     208500
1   2          20       RL         80.0       WD        Normal     181500
2   3          60       RL         68.0       WD        Normal     223500
3   4          70       RL         60.0       WD       Abnorml     140000


Observe the following:

1. The 1st feature is the sample ID which is irrelevant for model training purposes
1. The dataset contains both numerical and categorical data
1. The output label is numeric so this is a regression problem

Let's check for columns with missing values denoted by `NA` in our training data.

In [7]:
train_df_na_columns = train_df.columns[train_df.isna().any()].tolist()
train_df_na_columns, len(train_df_na_columns)

(['LotFrontage',
  'Alley',
  'MasVnrType',
  'MasVnrArea',
  'BsmtQual',
  'BsmtCond',
  'BsmtExposure',
  'BsmtFinType1',
  'BsmtFinType2',
  'Electrical',
  'FireplaceQu',
  'GarageType',
  'GarageYrBlt',
  'GarageFinish',
  'GarageQual',
  'GarageCond',
  'PoolQC',
  'Fence',
  'MiscFeature'],
 19)

Our training set contains `NA` values across 19 columns. We'll need to pre-process our data appropriately to avoid issues during the training process related to `NA` values.

1. For numerical columns, replace `NA` values with the mean of the remaining entries
1. For categorical columns,  treat `NA` as its own distinct category

Furthermore, we'll use scikit-learn's [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to transform only numerical columns to have zero mean and unit variance. The transformation will be applied only after we have pre-processed and cleaned the data with Pandas.

First, we'll need to install version 1.8.0 of scikit-learn.

In [8]:
%pip install scikit-learn==1.8.0


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Define the StandardScaler for our input features. We'll use it later once the data is properly pre-processed and cleaned with Pandas.

In [9]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
feature_scaler

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


For our output label, we'll use a hybrid approach:

1. Apply `log1p` transformation: $\log(1 + y_i)$
1. Apply StandardScaler to the result in \(1\)

This requires us to define a custom scaler class inheriting [`BaseEstimator`](https://scikit-learn.org/stable/modules/generated/sklearn.base.BaseEstimator.html) and [`TransformerMixin`](https://scikit-learn.org/stable/modules/generated/sklearn.base.TransformerMixin.html).

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin

class LogStandardScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.scaler = StandardScaler()

    def fit(self, y):
        y_log1p = np.log1p(y)
        self.scaler.fit(y_log1p)
        return self

    def transform(self, y):
        y_log1p = np.log1p(y)
        y_scaled = self.scaler.transform(y_log1p)
        return y_scaled

    def inverse_transform(self, y_scaled):
        y_log1p = self.scaler.inverse_transform(y_scaled)
        y = np.expm1(y_log1p)
        return y

label_scaler = LogStandardScaler()
label_scaler

LogStandardScaler()

Now for the data pre-processing and scaling pipeline. We'll pass both our training and test data through the same pipeline defined as a reusable function. Furthermore, only our training set includes the output labels so let's focus on processing the input features only in our pipeline.

In [11]:
import numpy as np

train_onehot_columns = None

def preprocess_and_scale(X_dataframe, is_train=False):
    global train_onehot_columns
    
    """
    Split input features by numeric and non-numeric columns
    """
    X_dataframe_numeric = X_dataframe.select_dtypes(include=['number'])
    X_dataframe_categoric = X_dataframe.select_dtypes(exclude=['number'])

    """
    For numeric data, fill NA values with the column mean
    """
    X_dataframe_numeric = X_dataframe_numeric.fillna(X_dataframe_numeric.mean(numeric_only=True))

    """
    For non-numeric data, apply one-hot encoding with the caveats below.
    1. Discard the 1st category to avoid collinearity
    2. Treat NA values as its own distinct category
    3. For training data, record the resulting columns from one-hot encoding
    4. For test data, reindex based on (3) to ensure training and test data have the same number of input features
    """
    X_dataframe_categoric = pd.get_dummies(X_dataframe_categoric, drop_first=True, dummy_na=True)
    if is_train:
        train_onehot_columns = X_dataframe_categoric.columns
    else:
        X_dataframe_categoric = X_dataframe_categoric.reindex(columns=train_onehot_columns, fill_value=0)

    """
    Convert numeric and non-numeric input features to separate Numpy arrays
    """
    X_numeric = X_dataframe_numeric.to_numpy()
    X_categoric = X_dataframe_categoric.to_numpy()

    """
    Apply StandardScaler to numeric input features only with the caveats below.
    1. For training data, use `fit_transform` to compute the mean and stddev then normalize
    2. For test data, use `transform` to normalize based on the pre-computed mean and stddev of the training data
    """
    if is_train:
        X_numeric = feature_scaler.fit_transform(X_numeric)
    else:
        X_numeric = feature_scaler.transform(X_numeric)

    """
    Combine and return both numeric and non-numeric features as 1 ndarray
    """
    return np.concatenate((X_numeric, X_categoric), axis=1)

In [12]:
X_train_df = train_df.iloc[:, :-1]
X_train_df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,8,2007,WD,Normal
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal


In [13]:
y_train_df = train_df.iloc[:, -1]
y_train_df

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: SalePrice, Length: 1460, dtype: int64

In [14]:
X_train = preprocess_and_scale(X_dataframe=X_train_df, is_train=True)
X_test = preprocess_and_scale(X_dataframe=test_df, is_train=False)
X_train.shape, X_test.shape

((1460, 288), (1459, 288))

Our pre-processed and transformed data has $288$ features and $1$ label. Let's transform our labels as well with the `LogStandardScaler` we defined earlier.

In [15]:
y_train = y_train_df.to_numpy().reshape(-1, 1)
y_train = label_scaler.fit_transform(y_train)
y_train.shape

(1460, 1)

## 5.7.5. Error Measure

The Kaggle competition uses [RMSLE](https://www.kaggle.com/code/carlolepelaars/understanding-the-metric-rmsle) as the loss function for evaluating submissions. RMSLE stands for "root mean squared logarithmic error" and is a variation of MSE loss with the following caveats.

1. We compute the difference between the logarithm of predicted vs. actual labels, instead of computing the difference between them directly
1. We take the square root of the resulting mean value as our loss

To train our deep neural network, we'll use MSE loss directly. Since we already log-transformed our output labels, it's functionally equivalent to computing the MSLE on the original labels.

## 5.7.6. $K$-Fold Cross-Validation

Our dataset has $80$ dimensions prior to pre-processing and transformation. However, we only have $1460$ samples for training.

To fully utilize our limited training samples, we'll use $k$-fold cross-validation with $k = 10$. This splits our training data into $10$ subsets of $146$ samples and we construct $10$ training and validation sets with the same size as our original training data \($1460$ samples\).

Each derived training and validation set has $146 \times 9 = 1314$ training and $146 \times 1 = 146$ validation samples respectively. Each time, a different subset of $146$ validation samples is used.

scikit-learn's [`sklearn.model_selection.KFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) provides us a convenient method to generate Numpy indices for $k$-fold splitting.

In [16]:
from sklearn.model_selection import KFold
import mindspore.dataset as ds

kf = KFold(n_splits=10, shuffle=True)
kf_ds = [{
    'train': ds.NumpySlicesDataset(data=(X_train[train_idx], y_train[train_idx]), column_names=['feature', 'label']),
    'val': ds.NumpySlicesDataset(data=(X_train[val_idx], y_train[val_idx]), column_names=['feature', 'label'])
} for (train_idx, val_idx) in kf.split(X_train)]
kf_ds_samples_count = [{ f'{k}_samples': v.get_dataset_size() for k, v in kf_dict.items() } for kf_dict in kf_ds]
kf_ds = [{ k: v.batch(batch_size=64, drop_remainder=False) for k, v in kf_dict.items() } for kf_dict in kf_ds]
len(kf_ds), kf_ds_samples_count

(10,
 [{'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146},
  {'train_samples': 1314, 'val_samples': 146}])

## 5.7.7. Model Selection

TODO